# Pass 2: The 7 Basic Emotions + Context
LLM coding of basic emotions in YouTube crisis comments.
Batch-by-batch processing per crisis event.

## Methodology

### Motivation
Pass 1 captures moral emotions (grief/outrage). Pass 2 adds a
psychology-anchored layer: Anger, Contempt, Disgust, Fear, Happiness,
Sadness, Surprise — plus the same grounds + family context for joining
across passes.

### Approach
Same chunked protocol with `../prompts/pass-02-basic-emotions.md`.
`basic_emotions` strictly subset of the 7; `[]` when unclear.
`value_or_relationship_invoked` / `family_referenced[]` same semantics as Pass 1.

### Pipeline

| Phase | Step | Description |
|---|---|---|
| **1–4** | Same as Pass 1 | Load → slim/clean → chunk (15) → LLM call + `json.loads` |
| **5. Export** | Save + inspect | `sample_pass02_results.jsonl` → merge on `row_id` for QA |

### Cross-Analysis
- *How do basic emotions map onto grief/outrage (join on `row_id`)?*
- *Which values co-occur with Contempt vs Anger vs Sadness?*

### Input / Output
- **Input:** `01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv`
- **Output:** `03-llm-passes/output/{crisis_name}/sample_pass02_results.jsonl`

## Setup
Import dependencies: `pandas` for data, `dotenv` + `google.genai` for LLM
calls, `json/re/time/pathlib` for chunking, parsing, and checkpointing.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai

## Configuration
Micro-sample selector. Papermill injects `crisis_name` / `phase_name` per run.

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "04-reckoning"  # default; overridden by papermill
n_sample = 30
chunk_size = 15
sample_seed = 42
model = "gemma-2-9b-it"
overwrite = False

In [ ]:
# ── Environment & API Setup ─────────────────────────────────────────────
load_dotenv("../../.env")
GEMMA_API_KEY = os.environ.get("GEMMA_API_KEY", "")

if not GEMMA_API_KEY:
    raise RuntimeError("GEMMA_API_KEY missing. Copy .env.example → .env and fill it.")
print("✅ Gemma API key loaded.")
client = genai.Client(api_key=GEMMA_API_KEY)

# ── Path Configurations ─────────────────────────────────────────────────
INPUT_PATH  = f"../../01-preprocessing/output/{crisis_name}/youtube/{phase_name}.csv"
OUTPUT_PATH = f"../output/{crisis_name}/sample_pass02_results.jsonl"
LOG_PATH    = f"../output/{crisis_name}/sample_pass02_failures.log"

# ── Model & Processing Configurations ───────────────────────────────────
MODEL       = model  # "gemma-2-9b-it" via parameters
TEMPERATURE = 0.0
MAX_TOKENS  = 512
BATCH_SIZE  = chunk_size  # 15 default from parameters
WORKERS     = 5           # concurrent threads (used when threaded runner lands)